# 🧲 The Stern-Gerlach Experiment
## Quantum Spin and Space Quantization

The Stern-Gerlach experiment (1922) was a landmark demonstration that **angular momentum is quantized** in quantum mechanics. Silver atoms were sent through an inhomogeneous magnetic field and, rather than spreading continuously, they split into **discrete beams** — direct evidence of quantized spin.

---

### Physical Setup

A beam of neutral atoms travels along the $y$-axis through a magnetic field $\mathbf{B}$ oriented along the $z$-axis, with a strong gradient $\partial B_z / \partial z$. The force on a magnetic dipole is:

$$F_z = \mu_z \frac{\partial B_z}{\partial z}$$

For a particle with spin quantum number $s$, the $z$-component of the magnetic moment takes **discrete values**:

$$\mu_z = -g_s \, \mu_B \, m_s, \qquad m_s = -s, -s+1, \ldots, +s$$

where $\mu_B = e\hbar / 2m_e$ is the Bohr magneton and $g_s \approx 2$ for the electron.

This leads to $2s+1$ discrete deflections, e.g. **2 spots** for spin-1/2 particles.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import matplotlib.gridspec as gridspec
from matplotlib.patches import FancyArrowPatch, Rectangle, Ellipse
from matplotlib.colors import LinearSegmentedColormap
from ipywidgets import interact, FloatSlider, IntSlider, Dropdown, HBox, VBox, Label
import warnings
warnings.filterwarnings('ignore')

# Physical constants
mu_B   = 9.2740100783e-24   # J/T  Bohr magneton
hbar   = 1.054571817e-34    # J·s
m_Ag   = 1.7924e-25         # kg   mass of silver atom
g_s    = 2.002319           # electron g-factor

print("Constants loaded.")
print(f"  Bohr magneton  μ_B = {mu_B:.4e} J/T")
print(f"  ℏ              = {hbar:.4e} J·s")
print(f"  Mass of Ag     = {m_Ag:.4e} kg")
print(f"  g_s (electron) = {g_s:.6f}")

---
## 1 · Schematic Diagram of the Apparatus

In [ ]:
fig, ax = plt.subplots(figsize=(14, 6))
ax.set_xlim(0, 14)
ax.set_ylim(-3, 3)
ax.set_aspect('equal')
ax.axis('off')
ax.set_facecolor('#0d1117')
fig.patch.set_facecolor('#0d1117')

# ── Oven ──────────────────────────────────────────────────────────────────────
oven = Rectangle((0.3, -0.6), 1.2, 1.2, linewidth=2,
                 edgecolor='#e07b39', facecolor='#5c2e00')
ax.add_patch(oven)
ax.text(0.9, 0, 'Oven\n(Ag)', color='#e07b39', ha='center', va='center',
        fontsize=9, fontweight='bold')

# collimator slit
ax.plot([1.7, 2.3], [-0.08, -0.08], color='#aaaaaa', lw=3)
ax.plot([1.7, 2.3], [ 0.08,  0.08], color='#aaaaaa', lw=3)
ax.text(2.0, -0.45, 'Collimator', color='#aaaaaa', ha='center', fontsize=8)

# ── Beam ──────────────────────────────────────────────────────────────────────
ax.annotate('', xy=(3.5, 0), xytext=(2.3, 0),
            arrowprops=dict(arrowstyle='->', color='#f0e68c', lw=1.5))

# ── Magnet poles ──────────────────────────────────────────────────────────────
# N pole (top, pointed)
N_pole = plt.Polygon([[3.5, 0.25],[3.5, 2.2],[7.5, 2.2],[7.5, 0.25],[5.5, 0.08]],
                     closed=True, facecolor='#1a3a6e', edgecolor='#4a8fd4', lw=2)
ax.add_patch(N_pole)
ax.text(5.5, 1.5, 'N', color='#4a8fd4', ha='center', va='center',
        fontsize=22, fontweight='bold')

# S pole (bottom, flat)
S_pole = Rectangle((3.5, -2.2), 4.0, 1.95,
                   facecolor='#6e1a1a', edgecolor='#d44a4a', lw=2)
ax.add_patch(S_pole)
ax.text(5.5, -1.5, 'S', color='#d44a4a', ha='center', va='center',
        fontsize=22, fontweight='bold')

# field gradient arrows inside magnet
for y in np.linspace(-0.18, 0.0, 3):
    ax.annotate('', xy=(5.5, -0.25), xytext=(5.5, 0.25),
                arrowprops=dict(arrowstyle='->', color='#88ccff', lw=0.8, alpha=0.5))
ax.text(6.6, 0.0, r'$\nabla B$', color='#88ccff', fontsize=11, va='center')

# ── Deflected beams ──────────────────────────────────────────────────────────
beam_colors = ['#ff6b6b', '#f0e68c']
deflections = [0.55, -0.55]
labels_beam = [r'$m_s = +\frac{1}{2}$', r'$m_s = -\frac{1}{2}$']

for defl, col, lab in zip(deflections, beam_colors, labels_beam):
    # inside magnet
    ax.annotate('', xy=(7.5, defl*0.6), xytext=(3.5, 0),
                arrowprops=dict(arrowstyle='->', color=col, lw=2))
    # after magnet
    ax.annotate('', xy=(11.5, defl*1.5), xytext=(7.5, defl*0.6),
                arrowprops=dict(arrowstyle='->', color=col, lw=2))
    ax.text(11.8, defl*1.5, lab, color=col, va='center', fontsize=9)

# ── Detector screen ──────────────────────────────────────────────────────────
screen = Rectangle((12.0, -2.0), 0.15, 4.0,
                   facecolor='#333333', edgecolor='#888888', lw=2)
ax.add_patch(screen)
ax.text(12.08, -2.3, 'Detector\nscreen', color='#888888',
        ha='center', fontsize=8)

# spots on screen
for defl, col in zip(deflections, beam_colors):
    spot = Ellipse((12.07, defl*1.5), 0.1, 0.25,
                  facecolor=col, alpha=0.9)
    ax.add_patch(spot)

# ── Labels ────────────────────────────────────────────────────────────────────
ax.text(5.5, 2.55, 'Inhomogeneous Magnetic Field Region', color='#cccccc',
        ha='center', fontsize=9, style='italic')
ax.text(7.0, -2.6, 'y  →  (beam direction)', color='#666666',
        ha='center', fontsize=8)
ax.annotate('', xy=(0.5, -2.4), xytext=(0.5, -1.9),
            arrowprops=dict(arrowstyle='->', color='#888888', lw=1.5))
ax.text(0.9, -2.15, 'z', color='#888888', fontsize=10)

ax.set_title('Stern–Gerlach Apparatus (Spin-½ particle)',
             color='white', fontsize=14, pad=10)
plt.tight_layout()
plt.show()

---
## 2 · Deflection Formula and Energy Levels

The deflection $\Delta z$ of an atom after traversing the magnet of length $L$ and then traveling a free distance $D$ to the screen is:

$$\Delta z = \frac{F_z}{m} \left(\frac{L^2}{2 v_0^2} + \frac{L\,D}{v_0^2}\right)$$

where $v_0$ is the initial beam velocity (taken from the Maxwell-Boltzmann distribution peak, $v_0 = \sqrt{2k_BT/m}$).

The Zeeman energy splitting for spin $s$ in field $B_0$ is:

$$E_{m_s} = g_s \, \mu_B \, m_s \, B_0$$

In [ ]:
k_B = 1.380649e-23  # J/K

def peak_velocity(T, mass):
    """Most probable speed from Maxwell-Boltzmann distribution."""
    return np.sqrt(2 * k_B * T / mass)

def deflection(m_s, dBdz, L, D, T, mass=m_Ag, g=g_s):
    """Vertical deflection on screen (metres)."""
    F  = g * mu_B * m_s * dBdz          # force (N)
    v0 = peak_velocity(T, mass)          # most probable speed (m/s)
    a  = F / mass                        # acceleration (m/s²)
    dz = a * (L**2 / (2*v0**2) + L*D / v0**2)
    return dz

def zeeman_energies(spin, B0, g=g_s):
    """Return array of Zeeman energy levels in eV."""
    ms_vals = np.arange(-spin, spin+1, 1.0)
    E_J  = g * mu_B * ms_vals * B0
    return ms_vals, E_J / 1.602176634e-19   # convert to eV

# Quick sanity check
T0     = 1000    # K
dBdz0  = 20      # T/m
L0     = 0.07    # m
D0     = 0.35    # m

v0 = peak_velocity(T0, m_Ag)
dz_up   = deflection(+0.5, dBdz0, L0, D0, T0) * 1e3
dz_down = deflection(-0.5, dBdz0, L0, D0, T0) * 1e3

print(f"Peak velocity at T={T0} K : {v0:.0f} m/s")
print(f"Deflection  m_s = +½  : {dz_up:+.2f} mm")
print(f"Deflection  m_s = -½  : {dz_down:+.2f} mm")
print(f"Separation (2Δz)      : {abs(dz_up - dz_down):.2f} mm")

---
## 3 · Interactive: Beam Deflection vs. Field Gradient

In [ ]:
def plot_deflection_interactive(dBdz=20.0, T=1000.0, L=0.07, D=0.35, spin=0.5):
    spin = float(spin)
    ms_vals = np.arange(-spin, spin + 1, 1.0)
    n_beams = len(ms_vals)

    colors = plt.cm.RdYlBu(np.linspace(0.05, 0.95, n_beams))

    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    fig.patch.set_facecolor('#0d1117')
    for ax in axes:
        ax.set_facecolor('#161b22')
        for spine in ax.spines.values():
            spine.set_color('#444')
        ax.tick_params(colors='#ccc')
        ax.xaxis.label.set_color('#ccc')
        ax.yaxis.label.set_color('#ccc')
        ax.title.set_color('white')

    # ── Left: deflection on screen ──────────────────────────────────────────
    ax = axes[0]
    screen_x = 0.08
    ax.axvline(0, color='#f0e68c', lw=2, ls='--', alpha=0.5, label='Undeflected beam')

    for ms, col in zip(ms_vals, colors):
        dz = deflection(ms, dBdz, L, D, T) * 1e3   # mm
        ax.plot([0, screen_x], [0, dz], color=col, lw=2.5,
                label=fr'$m_s={ms:+.1f}$,  $\Delta z={dz:+.2f}$ mm')
        ax.plot(screen_x, dz, 'o', color=col, ms=12, zorder=5)

    ax.set_xlabel('Propagation axis (arb.)', fontsize=11)
    ax.set_ylabel('Deflection Δz  (mm)', fontsize=11)
    ax.set_title(f'Beam splitting on detector  (spin = {spin})', fontsize=12)
    ax.legend(fontsize=8, facecolor='#1e2530', labelcolor='white', loc='upper left')
    ax.grid(True, alpha=0.15)

    # ── Right: Zeeman energy levels ──────────────────────────────────────────
    ax = axes[1]
    B_arr = np.linspace(0, 2.0, 300)

    for ms, col in zip(ms_vals, colors):
        E = g_s * mu_B * ms * B_arr / 1.602176634e-19 * 1e6   # μeV
        ax.plot(B_arr, E, color=col, lw=2.5, label=fr'$m_s={ms:+.1f}$')

    ax.axvline(1.0, color='white', lw=1, ls=':', alpha=0.4)
    ax.set_xlabel('Magnetic field  B  (T)', fontsize=11)
    ax.set_ylabel(r'Zeeman energy  $E_{m_s}$  (μeV)', fontsize=11)
    ax.set_title(f'Zeeman splitting  (spin = {spin})', fontsize=12)
    ax.legend(fontsize=8, facecolor='#1e2530', labelcolor='white')
    ax.grid(True, alpha=0.15)

    plt.suptitle(
        fr'$\partial B/\partial z$ = {dBdz:.0f} T/m,  T = {T:.0f} K,  '
        fr'L = {L*100:.0f} cm,  D = {D*100:.0f} cm',
        color='white', fontsize=11, y=1.01
    )
    plt.tight_layout()
    plt.show()

interact(
    plot_deflection_interactive,
    dBdz  = FloatSlider(value=20, min=1, max=100, step=1,
                        description='∂B/∂z (T/m)', style={'description_width':'130px'}),
    T     = FloatSlider(value=1000, min=300, max=3000, step=50,
                        description='Oven T (K)', style={'description_width':'130px'}),
    L     = FloatSlider(value=0.07, min=0.01, max=0.30, step=0.01,
                        description='Magnet L (m)', style={'description_width':'130px'}),
    D     = FloatSlider(value=0.35, min=0.05, max=1.0, step=0.05,
                        description='Drift D (m)', style={'description_width':'130px'}),
    spin  = Dropdown(options=[('½  (electron/Ag)', 0.5), ('1  (deuteron)', 1.0),
                               ('3/2', 1.5), ('2', 2.0), ('5/2', 2.5), ('3', 3.0)],
                     value=0.5, description='Spin s', style={'description_width':'130px'}),
);

---
## 4 · Maxwell–Boltzmann Velocity Distribution

Atoms leave the oven with a **Maxwell–Boltzmann** speed distribution. This thermal spread broadens each spot on the detector. The distribution is:

$$f(v) = 4\pi n \left(\frac{m}{2\pi k_B T}\right)^{3/2} v^2 \exp\!\left(-\frac{mv^2}{2k_BT}\right)$$

In [ ]:
def mb_distribution(v, T, mass):
    """Maxwell-Boltzmann speed distribution (normalized)."""
    a = np.sqrt(k_B * T / mass)
    return (np.sqrt(2/np.pi) * v**2 / a**3) * np.exp(-v**2 / (2*a**2))

def plot_velocity_and_spots(T=1000.0, dBdz=20.0, L=0.07, D=0.35):
    v_arr = np.linspace(1, 3000, 2000)
    fv    = mb_distribution(v_arr, T, m_Ag)
    v_mp  = peak_velocity(T, m_Ag)

    # compute deflection for each speed
    F_up   = g_s * mu_B * 0.5 * dBdz
    F_down = -F_up
    a_up   = F_up / m_Ag

    dz_up_arr   = (a_up * (L**2 / (2*v_arr**2) + L*D / v_arr**2)) * 1e3  # mm
    dz_down_arr = -dz_up_arr

    fig, axes = plt.subplots(1, 3, figsize=(16, 5))
    fig.patch.set_facecolor('#0d1117')
    for ax in axes:
        ax.set_facecolor('#161b22')
        for spine in ax.spines.values(): spine.set_color('#444')
        ax.tick_params(colors='#ccc')
        ax.xaxis.label.set_color('#ccc')
        ax.yaxis.label.set_color('#ccc')
        ax.title.set_color('white')

    # ── (a) MB distribution ──────────────────────────────────────────────────
    ax = axes[0]
    ax.fill_between(v_arr, fv, alpha=0.3, color='#4a8fd4')
    ax.plot(v_arr, fv, color='#4a8fd4', lw=2)
    ax.axvline(v_mp, color='#f0e68c', lw=2, ls='--', label=f'$v_{{mp}}$ = {v_mp:.0f} m/s')
    ax.set_xlabel('Speed  v  (m/s)', fontsize=11)
    ax.set_ylabel('f(v)  (normalized)', fontsize=11)
    ax.set_title(f'Maxwell–Boltzmann  (T = {T:.0f} K)', fontsize=11)
    ax.legend(facecolor='#1e2530', labelcolor='white', fontsize=9)
    ax.grid(True, alpha=0.15)

    # ── (b) deflection vs speed ───────────────────────────────────────────────
    ax = axes[1]
    ax.plot(v_arr, dz_up_arr,   color='#ff6b6b', lw=2, label=r'$m_s=+\frac{1}{2}$')
    ax.plot(v_arr, dz_down_arr, color='#4a8fd4', lw=2, label=r'$m_s=-\frac{1}{2}$')
    ax.axvline(v_mp, color='#f0e68c', lw=1.5, ls='--', alpha=0.7)
    ax.set_xlabel('Speed  v  (m/s)', fontsize=11)
    ax.set_ylabel('Deflection Δz  (mm)', fontsize=11)
    ax.set_title('Deflection vs. atomic speed', fontsize=11)
    ax.set_xlim(200, 2500)
    ax.set_ylim(-8, 8)
    ax.legend(facecolor='#1e2530', labelcolor='white', fontsize=9)
    ax.grid(True, alpha=0.15)

    # ── (c) simulated detector pattern ───────────────────────────────────────
    ax = axes[2]

    # weight deflection histogram by MB distribution
    weights = mb_distribution(v_arr, T, m_Ag)
    weights /= weights.sum()

    bins = np.linspace(-8, 8, 200)
    ax.hist(dz_up_arr,   bins=bins, weights=weights, color='#ff6b6b',
            alpha=0.7, label=r'$m_s=+\frac{1}{2}$', density=True)
    ax.hist(dz_down_arr, bins=bins, weights=weights, color='#4a8fd4',
            alpha=0.7, label=r'$m_s=-\frac{1}{2}$', density=True)
    ax.set_xlabel('Position on screen  Δz  (mm)', fontsize=11)
    ax.set_ylabel('Relative intensity', fontsize=11)
    ax.set_title('Simulated detector pattern', fontsize=11)
    ax.legend(facecolor='#1e2530', labelcolor='white', fontsize=9)
    ax.grid(True, alpha=0.15)

    plt.suptitle(fr'∂B/∂z = {dBdz} T/m,  L = {L*100:.0f} cm,  D = {D*100:.0f} cm',
                 color='white', fontsize=11, y=1.02)
    plt.tight_layout()
    plt.show()

interact(
    plot_velocity_and_spots,
    T     = FloatSlider(value=1000, min=300, max=3000, step=50,
                        description='Oven T (K)', style={'description_width':'120px'}),
    dBdz  = FloatSlider(value=20, min=1, max=100, step=1,
                        description='∂B/∂z (T/m)', style={'description_width':'120px'}),
    L     = FloatSlider(value=0.07, min=0.01, max=0.30, step=0.01,
                        description='Magnet L (m)', style={'description_width':'120px'}),
    D     = FloatSlider(value=0.35, min=0.05, max=1.0, step=0.05,
                        description='Drift D (m)', style={'description_width':'120px'}),
);

---
## 5 · Spin State Visualization on the Bloch Sphere

A spin-½ state $|\psi\rangle = \cos(\theta/2)|\uparrow\rangle + e^{i\phi}\sin(\theta/2)|\downarrow\rangle$ maps to a point on the **Bloch sphere**. The Stern-Gerlach measurement along $\hat{z}$ projects onto the north or south pole.

In [ ]:
def plot_bloch(theta_deg=45.0, phi_deg=0.0):
    theta = np.radians(theta_deg)
    phi   = np.radians(phi_deg)

    # Bloch vector
    x = np.sin(theta) * np.cos(phi)
    y = np.sin(theta) * np.sin(phi)
    z = np.cos(theta)

    # probabilities
    P_up   = np.cos(theta/2)**2
    P_down = np.sin(theta/2)**2

    fig = plt.figure(figsize=(12, 5))
    fig.patch.set_facecolor('#0d1117')

    # ── 3D Bloch sphere ──────────────────────────────────────────────────────
    ax3 = fig.add_subplot(121, projection='3d')
    ax3.set_facecolor('#0d1117')

    # wireframe sphere
    u = np.linspace(0, 2*np.pi, 60)
    v = np.linspace(0, np.pi, 40)
    xs = np.outer(np.cos(u), np.sin(v))
    ys = np.outer(np.sin(u), np.sin(v))
    zs = np.outer(np.ones_like(u), np.cos(v))
    ax3.plot_wireframe(xs, ys, zs, color='#333', linewidth=0.4, alpha=0.5)

    # axes
    for vec, label, col in [([1,0,0],'x','#ff6b6b'), ([0,1,0],'y','#6bff6b'), ([0,0,1],'z','#6b9fff')]:
        ax3.quiver(0,0,0, *vec, length=1.3, color=col, linewidth=1.5, arrow_length_ratio=0.1)
        ax3.text(*(np.array(vec)*1.4), label, color=col, fontsize=10, ha='center')

    # poles
    ax3.text(0, 0, 1.5, r'$|\uparrow\rangle$  (+½)', color='#ff9966', fontsize=10, ha='center')
    ax3.text(0, 0,-1.5, r'$|\downarrow\rangle$  (−½)', color='#66aaff', fontsize=10, ha='center')

    # Bloch vector
    ax3.quiver(0,0,0, x,y,z, length=1.0, color='#f0e68c', linewidth=3,
               arrow_length_ratio=0.15, normalize=False)
    ax3.scatter([x],[y],[z], color='#f0e68c', s=80, zorder=5)

    # projection on z axis
    ax3.plot([x,x],[y,y],[z,0], color='#f0e68c', lw=1, ls='--', alpha=0.5)
    ax3.plot([x,0],[y,0],[0,0], color='#f0e68c', lw=1, ls='--', alpha=0.5)

    ax3.set_xlim(-1.3,1.3); ax3.set_ylim(-1.3,1.3); ax3.set_zlim(-1.3,1.3)
    ax3.set_title(fr'Bloch sphere  θ={theta_deg:.0f}°, φ={phi_deg:.0f}°',
                  color='white', fontsize=11)
    ax3.tick_params(colors='#666')
    for pane in [ax3.xaxis.pane, ax3.yaxis.pane, ax3.zaxis.pane]:
        pane.fill = False

    # ── Measurement probabilities ────────────────────────────────────────────
    ax2 = fig.add_subplot(122)
    ax2.set_facecolor('#161b22')
    for spine in ax2.spines.values(): spine.set_color('#444')
    ax2.tick_params(colors='#ccc')

    bars = ax2.bar([r'$P(m_s=+\frac{1}{2})$', r'$P(m_s=-\frac{1}{2})$'],
                   [P_up, P_down],
                   color=['#ff9966', '#66aaff'], edgecolor='white', linewidth=1.5,
                   width=0.5)
    ax2.set_ylim(0, 1.1)
    ax2.set_ylabel('Probability', color='#ccc', fontsize=12)
    ax2.set_title('SG Measurement Outcome Probabilities', color='white', fontsize=11)
    ax2.axhline(0.5, color='white', lw=1, ls=':', alpha=0.4)

    for bar, p in zip(bars, [P_up, P_down]):
        ax2.text(bar.get_x() + bar.get_width()/2, p + 0.03,
                 f'{p:.3f}', ha='center', color='white', fontsize=13, fontweight='bold')

    ax2.text(0.5, 0.92,
             fr'$|\psi\rangle = \cos(\theta/2)|\uparrow\rangle + e^{{i\phi}}\sin(\theta/2)|\downarrow\rangle$',
             transform=ax2.transAxes, ha='center', color='#f0e68c', fontsize=9)
    ax2.grid(True, alpha=0.15, axis='y')

    plt.tight_layout()
    plt.show()

interact(
    plot_bloch,
    theta_deg = FloatSlider(value=45, min=0, max=180, step=1,
                            description='θ (polar °)', style={'description_width':'110px'}),
    phi_deg   = FloatSlider(value=0,  min=0, max=360, step=5,
                            description='φ (azimuth °)', style={'description_width':'110px'}),
);

---
## 6 · Sequential Stern-Gerlach Experiments

A hallmark of quantum measurement: after selecting $m_s = +\frac{1}{2}$ with a first SG device along $\hat{z}$, rotating a second device by angle $\alpha$ and measuring again gives probabilities:

$$P(+) = \cos^2\!\left(\frac{\alpha}{2}\right), \qquad P(-) = \sin^2\!\left(\frac{\alpha}{2}\right)$$

This is a direct consequence of the **non-commutativity** of spin operators.

In [ ]:
alpha_arr = np.linspace(0, 2*np.pi, 500)
P_plus  = np.cos(alpha_arr/2)**2
P_minus = np.sin(alpha_arr/2)**2

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.patch.set_facecolor('#0d1117')
for ax in axes:
    ax.set_facecolor('#161b22')
    for spine in ax.spines.values(): spine.set_color('#444')
    ax.tick_params(colors='#ccc')
    ax.xaxis.label.set_color('#ccc')
    ax.yaxis.label.set_color('#ccc')
    ax.title.set_color('white')

# ── Probability vs rotation angle ─────────────────────────────────────────────
ax = axes[0]
ax.plot(np.degrees(alpha_arr), P_plus,  color='#ff6b6b', lw=2.5,
        label=r'$P(m_s=+\frac{1}{2}) = \cos^2(\alpha/2)$')
ax.plot(np.degrees(alpha_arr), P_minus, color='#4a8fd4', lw=2.5,
        label=r'$P(m_s=-\frac{1}{2}) = \sin^2(\alpha/2)$')
ax.fill_between(np.degrees(alpha_arr), P_plus,  alpha=0.15, color='#ff6b6b')
ax.fill_between(np.degrees(alpha_arr), P_minus, alpha=0.15, color='#4a8fd4')
ax.axhline(0.5, color='white', lw=1, ls=':', alpha=0.4)
ax.set_xlabel('Rotation angle α  (degrees)', fontsize=12)
ax.set_ylabel('Probability', fontsize=12)
ax.set_title('Sequential SG: 2nd device rotated by α', fontsize=12)
ax.set_xticks([0, 90, 180, 270, 360])
ax.legend(facecolor='#1e2530', labelcolor='white', fontsize=10)
ax.grid(True, alpha=0.15)

# ── Polar plot ────────────────────────────────────────────────────────────────
ax2 = fig.add_subplot(122, projection='polar')
ax2.set_facecolor('#161b22')
ax2.tick_params(colors='#aaa')
ax2.plot(alpha_arr, P_plus,  color='#ff6b6b', lw=2.5, label=r'$P(+)$')
ax2.plot(alpha_arr, P_minus, color='#4a8fd4', lw=2.5, label=r'$P(-)$')
ax2.fill(alpha_arr, P_plus,  alpha=0.15, color='#ff6b6b')
ax2.fill(alpha_arr, P_minus, alpha=0.15, color='#4a8fd4')
ax2.set_title('Polar: probability vs. α', color='white', pad=15, fontsize=11)
ax2.legend(loc='upper right', bbox_to_anchor=(1.3, 1.1),
           facecolor='#1e2530', labelcolor='white', fontsize=9)
ax2.grid(True, alpha=0.2, color='#555')
ax2.spines['polar'].set_color('#444')

plt.suptitle('Sequential Stern–Gerlach: Quantum non-commutativity',
             color='white', fontsize=13, y=1.01)
plt.tight_layout()
plt.show()

---
## 7 · Monte Carlo Simulation of the Full Experiment

In [ ]:
def simulate_sg(N=5000, dBdz=20.0, T=1000.0, L=0.07, D=0.35,
                beam_width_mm=0.5, spin=0.5):
    np.random.seed(42)
    spin = float(spin)
    ms_vals = np.arange(-spin, spin+1, 1.0)
    n_beams = len(ms_vals)
    colors   = plt.cm.RdYlBu(np.linspace(0.05, 0.95, n_beams))

    # sample speeds from MB distribution
    a_param = np.sqrt(k_B * T / m_Ag)
    v_samples = np.abs(np.random.normal(0, a_param, (N, 3)))
    speeds = np.linalg.norm(v_samples, axis=1)

    # assign spin states randomly (equal probability for each m_s)
    spin_idx = np.random.randint(0, n_beams, N)

    # transverse initial position (beam collimation)
    z0 = np.random.normal(0, beam_width_mm/3, N)

    # compute deflection
    F_unit = g_s * mu_B * dBdz
    dz_arr = np.array([
        F_unit * ms_vals[si] / m_Ag * (L**2 / (2*sp**2) + L*D / sp**2) * 1e3
        for si, sp in zip(spin_idx, speeds)
    ])
    z_final = z0 + dz_arr

    # ── Plotting ──────────────────────────────────────────────────────────────
    fig, axes = plt.subplots(1, 2, figsize=(14, 5),
                             gridspec_kw={'width_ratios': [3, 1]})
    fig.patch.set_facecolor('#0d1117')
    for ax in axes:
        ax.set_facecolor('#161b22')
        for spine in ax.spines.values(): spine.set_color('#444')
        ax.tick_params(colors='#ccc')
        ax.xaxis.label.set_color('#ccc')
        ax.yaxis.label.set_color('#ccc')
        ax.title.set_color('white')

    # scatter of atom trajectories (subsample)
    ax = axes[0]
    sub = min(N, 2000)
    idx = np.random.choice(N, sub, replace=False)
    for i, (ms, col) in enumerate(zip(ms_vals, colors)):
        mask = spin_idx[idx] == i
        if mask.sum() == 0: continue
        # trajectory lines
        for j in np.where(mask)[0][:300]:
            orig_j = idx[j]
            ax.plot([0, 1], [z0[orig_j], z_final[orig_j]],
                    color=col, alpha=0.03, lw=0.5)
        # final scatter
        ax.scatter(np.ones(mask.sum()), z_final[idx[mask]],
                   c=[col], alpha=0.3, s=2)

    ax.axvline(0, color='#f0e68c', lw=2, ls='--', alpha=0.6, label='Source')
    ax.axvline(1, color='#aaa', lw=2, ls='-', alpha=0.6, label='Screen')
    ax.set_xlabel('Propagation (arb.)', fontsize=11)
    ax.set_ylabel('z position  (mm)', fontsize=11)
    ax.set_title(f'Monte Carlo: {N} atoms,  spin = {spin}', fontsize=12)
    ax.legend(fontsize=9, facecolor='#1e2530', labelcolor='white')
    ax.grid(True, alpha=0.1)

    # histogram on screen
    ax = axes[1]
    bins = np.linspace(z_final.min()-1, z_final.max()+1, 80)
    for i, (ms, col) in enumerate(zip(ms_vals, colors)):
        mask = spin_idx == i
        ax.hist(z_final[mask], bins=bins, orientation='horizontal',
                color=col, alpha=0.7, label=fr'$m_s={ms:+.1f}$', density=True)
    ax.set_xlabel('Intensity', fontsize=11)
    ax.set_ylabel('z  (mm)', fontsize=11)
    ax.set_title('Detector pattern', fontsize=12)
    ax.legend(fontsize=8, facecolor='#1e2530', labelcolor='white')
    ax.grid(True, alpha=0.1)

    plt.tight_layout()
    plt.show()

interact(
    simulate_sg,
    N         = IntSlider(value=5000, min=500, max=20000, step=500,
                          description='N atoms', style={'description_width':'120px'}),
    dBdz      = FloatSlider(value=20, min=1, max=100, step=1,
                            description='∂B/∂z (T/m)', style={'description_width':'120px'}),
    T         = FloatSlider(value=1000, min=300, max=3000, step=50,
                            description='Oven T (K)', style={'description_width':'120px'}),
    L         = FloatSlider(value=0.07, min=0.01, max=0.30, step=0.01,
                            description='Magnet L (m)', style={'description_width':'120px'}),
    D         = FloatSlider(value=0.35, min=0.05, max=1.0, step=0.05,
                            description='Drift D (m)', style={'description_width':'120px'}),
    beam_width_mm = FloatSlider(value=0.5, min=0.1, max=3.0, step=0.1,
                                description='Beam width (mm)', style={'description_width':'120px'}),
    spin      = Dropdown(options=[('½  (Ag/electron)', 0.5), ('1', 1.0),
                                  ('3/2', 1.5), ('2', 2.0)],
                         value=0.5, description='Spin s', style={'description_width':'120px'}),
);

---
## 8 · Historical Context & Key Takeaways

| Year | Milestone |
|------|-----------|
| 1922 | Stern & Gerlach observe two discrete silver atom spots in Frankfurt |
| 1925 | Uhlenbeck & Goudsmit propose *electron spin* $s = \frac{1}{2}$ |
| 1927 | Phipps & Taylor repeat experiment for hydrogen, confirming spin-½ |
| 1932 | Discovery of neutron extends idea to nuclear spin |
| 1952 | Ramsey develops separated oscillatory fields (NMR), Nobel 1989 |

### Key physics demonstrated

1. **Space quantization** — $\mu_z$ takes only $2s+1$ discrete values, not a continuum.
2. **Half-integer spin** — the two spots for Ag imply $2s+1 = 2$, so $s = \frac{1}{2}$, which is *purely quantum mechanical* (no classical analog).
3. **Quantum measurement** — a single SG device *irreversibly collapses* the spin state onto an eigenstate.
4. **Non-commutativity** — sequential devices along different axes produce interference, underlining $[S_x, S_z] \neq 0$.

---
*Notebook created for BSc/MSc quantum mechanics courses. All parameters are adjustable via interactive widgets.*